<a href="https://colab.research.google.com/github/abd500253-coder/Machine_Learning_Journey/blob/main/01_Data_Preprocessing/Column%20Transformer/Column_Transformer_and_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Admission Prediction using Column Transformer and Pipeline
**Author:** Abdul Rehman Shafiq

This notebook demonstrates how to build a clean, end-to-end machine learning pipeline using `scikit-learn`. We will preprocess numeric and categorical data dynamically using `ColumnTransformer` and evaluate multiple classifiers (Logistic Regression, Decision Tree, and Random Forest) to predict student admissions.

In [21]:
import pandas as pd
import numpy as np

# scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 1. Load and Inspect the Data
We begin by loading the dataset and checking the data types and missing values. The dataset contains features such as `Age`, `Study_Hours`, `Family_Income`, `City`, and `Gender`.

In [22]:
df = pd.read_csv(r'/content/student_admission.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              30 non-null     int64  
 1   Study_Hours      29 non-null     float64
 2   Previous_Score   29 non-null     float64
 3   Family_Income    29 non-null     float64
 4   City             29 non-null     object 
 5   Gender           29 non-null     object 
 6   Internet_Access  29 non-null     object 
 7   Admission        30 non-null     object 
dtypes: float64(3), int64(1), object(4)
memory usage: 2.0+ KB


## 2. Train-Test Split
Next, we isolate our target variable (`Admission`) and split the data into training and testing sets to ensure an unbiased evaluation of our models.

In [23]:
X = df.drop(columns=['Admission'])
y = df['Admission']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (24, 7)
Testing data shape: (6, 7)


## 3. Define Feature Selectors and Preprocessing Pipelines
We use `make_column_selector` to dynamically separate numeric and categorical columns.
* **Numeric Pipeline:** Imputes missing values with the mean and standardizes the features.
* **Categorical Pipeline:** Imputes missing values with the most frequent category and applies one-hot encoding.

In [24]:
# Dynamically select numeric and categorical columns
numeric_features = make_column_selector(dtype_include='number')
categorical_features = make_column_selector(dtype_include='object')

# Pipeline for numeric features
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a single ColumnTransformer preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ],
    remainder='passthrough'
)

## 4. Model Training and Evaluation
We will iterate through a dictionary of classifiers, combining each one with our `preprocessor` into a final pipeline. This ensures data leakage is prevented during training and testing.

In [25]:
# Define the models to evaluate
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier()
}

results = {}

# Iterate through models, build pipeline, train, and evaluate
for name, model_instance in models.items():
    # Create the full pipeline
    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_instance)
    ])

    # Fit the pipeline on the training data
    model_pipeline.fit(X_train, y_train)

    # Make predictions on the testing data
    y_pred = model_pipeline.predict(X_test)

    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'{name} : {acc}')

# Extract and print the feature names generated by the transformations
print("\nGenerated Feature Names:")
print(preprocessor.fit(X_train, y_train).get_feature_names_out())

Logistic Regression : 1.0
Decision Tree : 1.0
Random Forest : 1.0

Generated Feature Names:
['num__Age' 'num__Study_Hours' 'num__Previous_Score' 'num__Family_Income'
 'cat__City_Chiniot' 'cat__City_Faisalabad' 'cat__City_Lahore'
 'cat__City_Sargodha' 'cat__Gender_Female' 'cat__Gender_Male'
 'cat__Internet_Access_No' 'cat__Internet_Access_Yes']


## 5. Visualizing the Final Pipeline Architecture
By calling the final fitted pipeline object in a Jupyter or Colab environment, we can interactively visualize the architecture of our `ColumnTransformer` and final estimator.

In [26]:
# Display the interactive HTML representation of the pipeline
# (Note: model_pipeline currently holds the Random Forest model from the end of the loop)
model_pipeline.fit(X_train, y_train)
model_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c3c461d6ad0>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7c3c45d89480>)])),
                ('model', RandomForestClassifier())])